# nvidia/Qwen3.8-Flash-Next-NVFP4 -> private Kaggle Model (run this ON Kaggle)

**Notebook settings: Accelerator = None (CPU), Internet = ON.** No GPU quota is used.

### Before running: attach two Kaggle Secrets, start the run from the editor
Add-ons -> Secrets -> attach `KAGGLE_USERNAME` and `KAGGLE_KEY`, then **Save Version -> Save & Run All** from the
notebook editor (a run started through the Kaggle API could not reach the secrets service on 2026-09-26).

### How it works (no disk)
The first attempt staged the checkpoint in `/tmp` and was killed by Kaggle's per-session disk quota, although `df`
showed 1.2 TB free. This version streams every file straight from Hugging Face into Kaggle's resumable upload URL
in 256 MiB chunks (RAM only, ~1 GB per worker, 3 workers). Each file is hashed while it streams and compared with
the pinned Hugging Face tree (LFS sha256 / git blob sha1); the model instance is created **only if all 24 files
match**. A dropped connection resumes from the byte the server confirmed.

### What gets published
`nvidia/Qwen3.8-Flash-Next-NVFP4` at revision `fc694b54fb…`, byte-identical: 10 NVFP4 shards,
`model-fp8-mtp-ple.safetensors` (MTP + PLE, 53.7 GB), index, tokenizer/processor configs, README, plus
`KAGGLE_MIRROR_MANIFEST.json` with the verified hashes (`.gitattributes` is dropped).

Kaggle path afterwards: `/kaggle/input/models/<user>/qwen3-8-flash-next-nvfp4-nvidia/pytorch/nvidia-modelopt-mixed/1`

The run is idempotent: if the model instance already exists it only lists its files.

In [ ]:
import hashlib
import json
import os
import queue
import shutil
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

HF_REPO = "nvidia/Qwen3.8-Flash-Next-NVFP4"
HF_REVISION = "fc694b54fb0174e0913e6adf86691ef85a4ead47"   # 2026-09-05, ModelOpt 0.46.0.dev281
HF_URL = "https://huggingface.co/{repo}/resolve/{rev}/{name}"
KAGGLE_MODEL_SLUG = "qwen3-8-flash-next-nvfp4-nvidia"
KAGGLE_MODEL_TITLE = "Qwen3.8 Flash-Next NVFP4 (NVIDIA)"
KAGGLE_FRAMEWORK = "pyTorch"          # appears as "pytorch" in the /kaggle/input path
KAGGLE_INSTANCE_SLUG = "nvidia-modelopt-mixed"
LICENSE_NAME = "Other (specified in description)"   # NVIDIA Open Model License; same label as B0's model
PLACEHOLDER_DIR = Path("/tmp/nv-instance-placeholders")   # empty files only: names for model_instance_create
CHUNK = 256 << 20          # bytes per upload request (multiple of 256 KiB, as the resumable protocol requires)
QUANTUM = 256 << 10
READ = 8 << 20             # Hugging Face read size
WORKERS = 3

# name -> (bytes, hash kind, hash). "sha256" = LFS oid, "gitsha1" = git blob id (non-LFS files).
EXPECTED_FILES = {
    '.gitattributes': (1635, 'gitsha1', 'a09db2ea4d1bd1fc1c09f6fca45e8e4054953535'),
    'README.md': (12109, 'gitsha1', 'f3edfc5b8f2cae8ae751162626d4c5aa3ed071a6'),
    'chat_template.jinja': (8952, 'gitsha1', 'c0c686f9c38d70d179fb7b5f5aa7530bc913dda3'),
    'config.json': (30820, 'gitsha1', 'd9df7636dc5b8bdab82655aba8ce0ebe25b22342'),
    'generation_config.json': (202, 'gitsha1', '023756cfadf88e5bf69eefeee3e172f38c448d64'),
    'hf_quant_config.json': (23001, 'gitsha1', '89bb1a48d2358f4b9ab7959bd4357a6d6d04020e'),
    'merges.txt': (3353259, 'gitsha1', 'a494e019ca1502219fd0128658b979e5f05ae8e8'),
    'model-00001-of-00010.safetensors': (3115991696, 'sha256', '63fde954be6f08b49b876f4f70a0ad0bcfee71aff7b1faa33779b6b32feca2a2'),
    'model-00002-of-00010.safetensors': (10005510304, 'sha256', '4dafaef62a908e49e0d92c7c2a3fa99f4d9651fdeeb0ca09f09a9b40095af4e0'),
    'model-00003-of-00010.safetensors': (10005364728, 'sha256', '3218ddc129258e91a721a8e81329ca8f00588332ec721bd8d2cf7d20e324bf47'),
    'model-00004-of-00010.safetensors': (10006060112, 'sha256', '55e2bdf6a3a1f6e65270787f65c63b2a2a56b308b54eb1fb318dfabc9b48b141'),
    'model-00005-of-00010.safetensors': (10005362392, 'sha256', 'd3c169d3694bfba846455d01e48f047d770f78d071c5a4d22f129389411d686e'),
    'model-00006-of-00010.safetensors': (10005375328, 'sha256', '38f65a9d11090428e139cc60d0587e8e5433ea2ce79820883c583d7cfcc53130'),
    'model-00007-of-00010.safetensors': (10005956544, 'sha256', '8cf3fa05c04cb2e060963b69bb01f7a7b9f43c58435fbe380836a0457a19cd3e'),
    'model-00008-of-00010.safetensors': (10005375200, 'sha256', '53d1f80746aa0cc0a7c2f4836587002a8a4dd72c827432cd2e43abae3c3899e4'),
    'model-00009-of-00010.safetensors': (5679113808, 'sha256', 'eaff6a87ece6fbfd6ad0fbaa0a5cf9ff542f8076acba38e62410b1ed55e12cc3'),
    'model-00010-of-00010.safetensors': (128587536, 'sha256', '0d49b0cf7c15bf9b5bd1e17403b36727315860a8f2a2e0f5a53655547084f640'),
    'model-fp8-mtp-ple.safetensors': (53717551730, 'sha256', '3525520c8602d850003eb1960aec0b64291dae33b83f8b65dd639a451df78823'),
    'model.safetensors.index.json': (31275518, 'sha256', '660414e8300728ba80062e6a3c81cb76a65e2ab136ac9e9ceb450ba0c51c3c0d'),
    'preprocessor_config.json': (390, 'gitsha1', '2ea84a437d448ff71b08df68fdd949d5cc4ebb64'),
    'processor_config.json': (1191, 'gitsha1', '33818c7f9e991ad735fd240209f4fa73e6c28c50'),
    'tokenizer.json': (12809320, 'sha256', '0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3'),
    'tokenizer_config.json': (17928, 'gitsha1', '5de744b3fca2129d7186979ae47c06be33903243'),
    'video_preprocessor_config.json': (385, 'gitsha1', '3ba673a5ad7d4d13f54155ecd38b2a94a6dac8fe'),
    'vocab.json': (6722759, 'gitsha1', '0aa0ce0658d60ac4a5d609f4eadb0e8e43514176'),
}
EXPECTED_TOTAL_BYTES = 132_734_506_847
NOT_UPLOADED = {".gitattributes"}
assert sum(v[0] for v in EXPECTED_FILES.values()) == EXPECTED_TOTAL_BYTES
assert len(EXPECTED_FILES) == 25
print("python", sys.version.split()[0], "| cpus", os.cpu_count(),
      "| /tmp free (df, not the session quota):", round(shutil.disk_usage("/tmp").free / 1e9, 1), "GB")


In [ ]:
# Kaggle Secrets -> Kaggle API credentials. Stops here (before any transfer) if they are missing or wrong.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    KAGGLE_USERNAME = _secrets.get_secret("KAGGLE_USERNAME").strip()
    _key = _secrets.get_secret("KAGGLE_KEY").strip()
except Exception as exc:
    raise RuntimeError(
        f"Could not read Kaggle Secrets KAGGLE_USERNAME / KAGGLE_KEY ({exc!r}). Attach both (Add-ons -> Secrets) "
        "and start the run from the notebook editor (Save Version -> Save & Run All). A run started through the "
        "Kaggle API got ConnectionError from the secrets service on 2026-09-26.") from None

cred_dir = Path.home() / ".kaggle"
cred_dir.mkdir(parents=True, exist_ok=True)
(cred_dir / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": _key}))
(cred_dir / "kaggle.json").chmod(0o600)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = _key
del _key

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.4"])

from kaggle.api.kaggle_api_extended import KaggleApi
from kaggle.models.upload_file import UploadFile
from kagglesdk.blobs.types.blob_api_service import ApiBlobType, ApiStartBlobUploadRequest

api = KaggleApi()
api.authenticate()
MODEL_REF = f"{KAGGLE_USERNAME}/{KAGGLE_MODEL_SLUG}"
INSTANCE_REF = f"{MODEL_REF}/pytorch/{KAGGLE_INSTANCE_SLUG}"

# Credential check first: kernels_list(mine=True) answers 401 for a wrong key. (Get-calls cannot be used for
# this: Kaggle answers 403 PERMISSION_DENIED both for a missing model and for someone else's private one.)
try:
    api.kernels_list(mine=True, page_size=1)
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    raise RuntimeError(f"Kaggle rejected the KAGGLE_USERNAME / KAGGLE_KEY secrets (HTTP {status}). "
                       "Check the two secret values, then run again.") from None

model_exists = MODEL_REF in {getattr(m, "ref", None) for m in (api.model_list(owner=KAGGLE_USERNAME, page_size=100) or [])}
instance_exists = False
if model_exists:
    try:
        api.model_instance_get(INSTANCE_REF)
        instance_exists = True
    except Exception as exc:
        if getattr(getattr(exc, "response", None), "status_code", None) not in (403, 404):
            raise
print(f"Kaggle API OK as {KAGGLE_USERNAME}; model exists={model_exists}; instance exists={instance_exists}")
if instance_exists:
    print("The instance already exists: the transfer is skipped; the last cell lists its files.")


In [ ]:
# Stream every file Hugging Face -> Kaggle resumable upload, hashing on the fly. Nothing is written to disk.
class _HFReader:
    """Reads one file from a byte offset in a background thread into a bounded queue (<= 48 x 8 MiB)."""

    def __init__(self, url, start, size):
        self.q = queue.Queue(maxsize=48)
        self.stop = threading.Event()
        self.error = None
        self.thread = threading.Thread(target=self._run, args=(url, start, size), daemon=True)
        self.thread.start()

    def _run(self, url, start, size):
        try:
            headers = {"Accept-Encoding": "identity"}
            if start:
                headers["Range"] = f"bytes={start}-"
            with requests.get(url, headers=headers, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                if start and r.status_code != 206:
                    raise RuntimeError(f"Hugging Face ignored the Range header (HTTP {r.status_code})")
                got = start
                for piece in r.iter_content(READ):
                    if self.stop.is_set():
                        return
                    got += len(piece)
                    self.q.put(piece)
                if got != size:
                    raise RuntimeError(f"Hugging Face stream ended at {got} of {size} bytes")
        except BaseException as exc:
            self.error = exc
        finally:
            self.q.put(None)

    def get(self):
        piece = self.q.get()
        if piece is None:
            raise self.error or RuntimeError("Hugging Face stream ended early")
        return piece

    def close(self):
        self.stop.set()
        try:
            while True:
                self.q.get_nowait()
        except queue.Empty:
            pass


def _start_blob(name, size):
    req = ApiStartBlobUploadRequest()
    req.type = ApiBlobType.MODEL
    req.name = name
    req.content_length = size
    req.last_modified_epoch_seconds = int(time.time())
    with api.build_kaggle_client() as kaggle:
        resp = api.with_retry(kaggle.blobs.blob_api_client.start_blob_upload)(req)
    return resp.token, resp.create_url


def _persisted(response, size):
    """Bytes the upload server has stored, from a chunk or status response."""
    if response.status_code in (200, 201):
        return size
    if response.status_code == 308:
        rng = response.headers.get("Range")
        return int(rng.rsplit("-", 1)[1]) + 1 if rng else 0
    raise RuntimeError(f"upload server answered HTTP {response.status_code}: {response.text[:300]}")


def _query(session, put_url, size):
    r = session.put(put_url, headers={"Content-Range": f"bytes */{size}", "Content-Length": "0"}, timeout=(30, 120))
    return _persisted(r, size)


def stream_file(name, size, kind, max_stalls=12):
    """Returns (token, sha256 hex, git-sha1 hex or None, seconds, failures). Gives up only after max_stalls
    consecutive failures without progress (a 53.7 GB file is ~200 requests; isolated resets are expected)."""
    token, put_url = _start_blob(name, size)
    session = requests.Session()
    h256 = hashlib.sha256()
    gsha = hashlib.sha1(b"blob %d\0" % size) if kind == "gitsha1" else None
    hashed = persisted = 0          # hashed == bytes confirmed stored and folded into the digests
    failures = stalls = 0
    stall_mark = -1
    next_report = 5e9
    started = time.time()
    while hashed < size or persisted < size:
        reader = _HFReader(HF_URL.format(repo=HF_REPO, rev=HF_REVISION, name=name), hashed, size)
        buf = bytearray()
        buf_start = hashed
        try:
            while buf_start < size:
                while len(buf) < CHUNK and buf_start + len(buf) < size:
                    buf += reader.get()
                if persisted > buf_start:          # stored by an earlier attempt: hash only, do not resend
                    n = min(persisted - buf_start, len(buf))
                else:
                    at_end = buf_start + len(buf) == size
                    send = len(buf) if at_end else (len(buf) // QUANTUM) * QUANTUM
                    r = session.put(put_url, data=bytes(buf[:send]), timeout=(30, 900),
                                    headers={"Content-Range": f"bytes {buf_start}-{buf_start + send - 1}/{size}"})
                    persisted = _persisted(r, size)
                    n = persisted - buf_start
                    if n < 0 or n > len(buf):
                        raise RuntimeError(f"server reports {persisted} stored; chunk started at {buf_start}")
                with memoryview(buf) as mv:
                    h256.update(mv[:n])
                    if gsha is not None:
                        gsha.update(mv[:n])
                del buf[:n]
                buf_start += n
                hashed = buf_start
                if hashed >= next_report:
                    dt = time.time() - started
                    print(f"    {name}: {hashed / 1e9:.1f}/{size / 1e9:.1f} GB ({hashed / 1e6 / dt:.0f} MB/s)", flush=True)
                    next_report += 5e9
        except Exception as exc:
            failures += 1
            stalls = stalls + 1 if hashed == stall_mark else 1
            stall_mark = hashed
            print(f"    {name}: attempt failed at {hashed} bytes (stall {stalls}/{max_stalls}): {exc!r}"[:400], flush=True)
            if stalls >= max_stalls:
                raise
            session.close()
            session = requests.Session()          # drop pooled connections after a failure
            time.sleep(min(120, 10 * stalls))
            for q in range(5):
                try:
                    persisted = _query(session, put_url, size)
                    break
                except Exception as qexc:
                    if q == 4:
                        raise
                    print(f"    {name}: status query failed ({qexc!r}); retrying"[:300], flush=True)
                    time.sleep(15 * (q + 1))
            if persisted < hashed:
                raise RuntimeError(f"{name}: server now reports {persisted} bytes < {hashed} already confirmed")
        finally:
            reader.close()
    return token, h256.hexdigest(), gsha.hexdigest() if gsha is not None else None, time.time() - started, failures


def upload_bytes(name, data):
    token, put_url = _start_blob(name, len(data))
    r = requests.put(put_url, data=data, headers={"Content-Range": f"bytes 0-{len(data) - 1}/{len(data)}"},
                     timeout=(30, 300))
    if _persisted(r, len(data)) != len(data):
        raise RuntimeError(f"{name}: small upload incomplete")
    return token


TOKENS, HASHES = {}, {}
if not instance_exists:
    todo = sorted((n for n in EXPECTED_FILES if n not in NOT_UPLOADED), key=lambda n: -EXPECTED_FILES[n][0])
    t0 = time.time()

    def _one(name):
        size, kind, want = EXPECTED_FILES[name]
        print(f"[{time.strftime('%H:%M:%S')}] start {name} ({size / 1e9:.2f} GB)", flush=True)
        for attempt in range(1, 4):      # a fresh upload session per attempt (e.g. if the old one expired)
            try:
                token, s256, g1, dt, fails = stream_file(name, size, kind)
                break
            except Exception as exc:
                print(f"[{time.strftime('%H:%M:%S')}] {name}: file attempt {attempt}/3 failed: {exc!r}"[:400], flush=True)
                if attempt == 3:
                    raise
        got = s256 if kind == "sha256" else g1
        print(f"[{time.strftime('%H:%M:%S')}] done  {name} in {dt:.0f} s ({size / 1e6 / max(dt, 1e-6):.0f} MB/s, "
              f"{fails} retries) {kind} {'OK' if got == want else 'MISMATCH'}", flush=True)
        return name, token, s256, got == want

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for name, token, s256, ok in pool.map(_one, todo):
            TOKENS[name], HASHES[name] = token, (s256, ok)
    bad = sorted(n for n, (_, ok) in HASHES.items() if not ok)
    total = sum(EXPECTED_FILES[n][0] for n in todo)
    dt = time.time() - t0
    print(f"\nstreamed {len(todo)} files, {total / 1e9:.2f} GB in {dt / 60:.1f} min ({total / 1e6 / dt:.0f} MB/s)")
    if bad:
        raise RuntimeError(f"hash mismatch, instance NOT created: {bad}")
    print(f"all {len(todo)} files match the pinned Hugging Face revision byte-for-byte")

    mirror_manifest = {
        "source": {"hf_repo": HF_REPO, "hf_revision": HF_REVISION},
        "verified_against": "Hugging Face tree API at the pinned revision (LFS sha256 / git blob sha1), "
                            "hashed while streaming",
        "files": {n: {"bytes": EXPECTED_FILES[n][0], "sha256": HASHES[n][0]} for n in sorted(todo)},
        "total_bytes": total,
        "not_uploaded": sorted(NOT_UPLOADED),
        "serving_notes": {
            "architecture": "Qwen4ExpForConditionalGeneration (model_type qwen4_exp)",
            "quantization": "ModelOpt MIXED_PRECISION: routed experts NVFP4, MTP routed experts FP8 128x128 block "
                            "(FP8_PB_WO / FP8_BLOCK_SCALES), PLE n-gram embedding FP8; vLLM 0.30.0 resolves it to "
                            "modelopt_mixed",
            "vllm": ">= 0.30.0 (Qwen4Exp + PR #55513 MTP fix + PLE CPU offload default on)",
            "card_example": "--quantization modelopt --reasoning-parser qwen3 --trust-remote-code; "
                            "MTP: --speculative-config {method: mtp} --no-enable-flashinfer-autotune",
            "license": "NVIDIA Open Model License (see README.md)",
        },
    }
    TOKENS["KAGGLE_MIRROR_MANIFEST.json"] = upload_bytes(
        "KAGGLE_MIRROR_MANIFEST.json", (json.dumps(mirror_manifest, indent=1) + "\n").encode("utf-8"))


In [ ]:
# Publish: parent model (if new), then the instance, handing the already-uploaded blob tokens to the library.
if not instance_exists:
    if not model_exists:
        parent_dir = Path("/tmp/kaggle-model-parent")
        shutil.rmtree(parent_dir, ignore_errors=True)
        parent_dir.mkdir(parents=True)
        (parent_dir / "model-metadata.json").write_text(json.dumps({
            "ownerSlug": KAGGLE_USERNAME,
            "title": KAGGLE_MODEL_TITLE,
            "slug": KAGGLE_MODEL_SLUG,
            "subtitle": "NVIDIA ModelOpt NVFP4 of Qwen3.8-Flash-Next, for offline vLLM 0.30",
            "isPrivate": True,
            "description": (
                f"Private mirror of {HF_REPO} (revision {HF_REVISION}) for offline Kaggle inference. "
                "Mixed precision by NVIDIA ModelOpt: NVFP4 routed experts, FP8 MTP and PLE. "
                "License: NVIDIA Open Model License (see README.md in the files)."
            ),
            "licenseName": LICENSE_NAME,
        }, indent=2))
        created = api.model_create_new(str(parent_dir))
        if getattr(created, "error", None):
            raise RuntimeError(f"model create failed: {created.error}")
        print("model created:", getattr(created, "ref", None), getattr(created, "url", None))

    # model_instance_create walks a folder and asks _upload_file for a token per file. The folder holds empty
    # placeholders with the real names; the patched _upload_file returns the token of the streamed blob.
    shutil.rmtree(PLACEHOLDER_DIR, ignore_errors=True)
    PLACEHOLDER_DIR.mkdir(parents=True)
    for name in TOKENS:
        (PLACEHOLDER_DIR / name).touch()
    (PLACEHOLDER_DIR / "model-instance-metadata.json").write_text(json.dumps({
        "ownerSlug": KAGGLE_USERNAME,
        "modelSlug": KAGGLE_MODEL_SLUG,
        "instanceSlug": KAGGLE_INSTANCE_SLUG,
        "framework": KAGGLE_FRAMEWORK,
        "overview": (f"{HF_REPO} at {HF_REVISION}, uploaded byte-identical (hashes in "
                     "KAGGLE_MIRROR_MANIFEST.json). 10 NVFP4 shards + model-fp8-mtp-ple.safetensors."),
        "usage": ("Serve ${PATH} with vLLM >= 0.30.0 (modelopt_mixed, Qwen4Exp, MTP via PR #55513, "
                  "PLE CPU offload). Parsers: reasoning qwen3, tool-call qwen3_coder."),
        "licenseName": LICENSE_NAME,
        "fineTunable": False,
        "trainingData": [],
        "modelInstanceType": "Unspecified",
        "baseModelInstance": "",
        # "" exactly as in the proven Qwen/Occamy notebooks: the URL field belongs to the EXTERNAL_VARIANT type.
        "externalBaseModelUrl": "",
    }, indent=2))
    used = []
    _original_upload_file = KaggleApi._upload_file

    def _token_for(self, file_name, full_path, blob_type, upload_context, quiet, resources, content_type=None):
        if file_name not in TOKENS:
            raise RuntimeError(f"no streamed blob for {file_name}")
        used.append(file_name)
        return UploadFile(token=TOKENS[file_name])

    KaggleApi._upload_file = _token_for
    try:
        response = api.model_instance_create(str(PLACEHOLDER_DIR), quiet=True, dir_mode="skip")
    finally:
        KaggleApi._upload_file = _original_upload_file
    if sorted(used) != sorted(TOKENS):
        raise RuntimeError(f"token hand-off mismatch: used={sorted(used)}")
    if getattr(response, "error", None):
        raise RuntimeError(f"instance create failed: {response.error}")
    print(f"instance created with {len(used)} files:", getattr(response, "ref", None), getattr(response, "url", None))


In [ ]:
# List what Kaggle now holds (processing can lag a few minutes behind the create call).
expected_upload = {n: EXPECTED_FILES[n][0] for n in EXPECTED_FILES if n not in NOT_UPLOADED}
listed = {}
for attempt in range(20):
    try:
        resp = api.model_instance_version_files(f"{INSTANCE_REF}/1", page_size=100)
        listed = {f.name: f.size for f in (getattr(resp, "files", None) or [])}
    except Exception as exc:
        print("listing not ready:", repr(exc)[:200])
    if set(expected_upload) <= set(listed):
        break
    time.sleep(60)
print(f"{len(listed)} files listed for {INSTANCE_REF}/1")
for name in sorted(listed):
    print(f"  {listed[name]:>14,}  {name}")
missing = sorted(set(expected_upload) - set(listed))
wrong = {n: (listed[n], expected_upload[n]) for n in expected_upload if n in listed and listed[n] != expected_upload[n]}
print("missing:", missing or "none", "| size mismatches:", wrong or "none")
print("\nattach with:", f'"model_sources": ["{INSTANCE_REF}/1"]')
print("mount path :", f"/kaggle/input/models/{INSTANCE_REF}/1")
